# 16 · Clinical nomogram for 18-month pain worsening

Build a simple clinically usable scoring tool using logistic regression on a curated, pre-operatively available feature set, then `rms::nomogram`.

In [1]:
source("helpers/pain_helpers.R")
suppressPackageStartupMessages({ library(dplyr); library(rms); library(ggplot2); library(pROC) })
dat <- readRDS(file.path(OUT_OBJ, "patient_anchor_features.rds"))
cat("Columns:", paste(names(dat), collapse = ", "), "\n")

feat_cols <- c("age_at_visit", "duration_yrs", "updrs3_score", "LEDD",
               "pre_mean", "NP1DPRS", "scopa", "stai", "pre_slope")
median_impute <- function(x){ x[is.na(x)] <- stats::median(x, na.rm=TRUE); x }
dm <- dat %>% dplyr::select(will_receive_dbs, worsened, dplyr::all_of(feat_cols)) %>%
  dplyr::mutate(dbs = as.integer(will_receive_dbs),
                dplyr::across(dplyr::all_of(feat_cols), median_impute)) %>%
  dplyr::select(-will_receive_dbs)
cat("Analytic n:", nrow(dm), "  worsening rate:", round(mean(dm$worsened), 3), "\n")

Columns: PATNO, pre_mean, pre_max, pre_sd, pre_last, pre_n, pre_slope, post_mean, post_n, delta, worsened, will_receive_dbs, age_at_visit, ageonset, duration_yrs, SEX, fampd, BMI, LEDD, updrs3_score, NHY, NP1DPRS, NP1ANXS, gds, stai, scopa 


Analytic n: 642   worsening rate: 0.157 


In [2]:
dd <- rms::datadist(dm); options(datadist = "dd")
fit_lrm <- rms::lrm(
  worsened ~ dbs + age_at_visit + duration_yrs + updrs3_score + LEDD +
             pre_mean + NP1DPRS + scopa + stai + pre_slope,
  data = dm, x = TRUE, y = TRUE
)
print(fit_lrm)

Logistic Regression Model

rms::lrm(formula = worsened ~ dbs + age_at_visit + duration_yrs + 
    updrs3_score + LEDD + pre_mean + NP1DPRS + scopa + stai + 
    pre_slope, data = dm, x = TRUE, y = TRUE)

                      Model Likelihood       Discrimination    Rank Discrim.    
                            Ratio Test              Indexes          Indexes    
Obs           642    LR chi2     30.51       R2       0.080    C       0.660    
 0            541    d.f.           10      R2(10,642)0.031    Dxy     0.321    
 1            101    Pr(> chi2) 0.0007    R2(10,255.3)0.077    gamma   0.321    
max |deriv| 3e-11                            Brier    0.127    tau-a   0.085    

             Coef    S.E.   Wald Z Pr(>|Z|)
Intercept    -3.2307 0.9352 -3.45  0.0006  
dbs           0.0305 0.4167  0.07  0.9417  
age_at_visit  0.0095 0.0123  0.77  0.4434  
duration_yrs  0.1249 0.0676  1.85  0.0646  
updrs3_score  0.0133 0.0098  1.36  0.1735  
LEDD         -0.0005 0.0004 -1.30  0.1939  
p

In [3]:
nom <- rms::nomogram(fit_lrm, fun = plogis,
                    fun.at = c(0.05, 0.1, 0.15, 0.25, 0.40, 0.55, 0.70),
                    funlabel = "Probability of 18-month pain worsening")
png(file.path(OUT_FIG, "Figure10_nomogram.png"), width = 2500, height = 1800, res = 280)
plot(nom, cex.axis = 0.85, cex.var = 1.05)
invisible(dev.off())
cat("Nomogram saved.\n")

val <- rms::validate(fit_lrm, method = "boot", B = 200)
print(val)
saveRDS(val, file.path(OUT_OBJ, "nomogram_validation.rds"))

Nomogram saved.


          index.orig training    test optimism index.corrected   Lower  Upper
Dxy           0.3205   0.3677  0.2839   0.0839          0.2367  0.1246 0.3392
R2            0.0798   0.1053  0.0601   0.0451          0.0347 -0.0308 0.0871
Intercept     0.0000   0.0000 -0.4009   0.4009         -0.4009 -0.8891 0.1742
Slope         1.0000   1.0000  0.7417   0.2583          0.7417  0.4497 1.0995
Emax          0.0000   0.0000  0.1581  -0.1581          0.1581 -0.0270 0.3607
D             0.0460   0.0618  0.0340   0.0277          0.0182 -0.0232 0.0513
U            -0.0031  -0.0031  0.0043  -0.0074          0.0043 -0.0069 0.0259
Q             0.0491   0.0649  0.0298   0.0351          0.0140 -0.0486 0.0558
B             0.1266   0.1237  0.1291  -0.0054          0.1320  0.1151 0.1490
g             0.7196   0.8306  0.6022   0.2283          0.4912  0.1890 0.7556
gp            0.0864   0.0976  0.0744   0.0232          0.0631  0.0320 0.0929
            n
Dxy       200
R2        200
Intercept 200
Slope   

In [4]:
cal <- rms::calibrate(fit_lrm, method = "boot", B = 200)
png(file.path(OUT_FIG, "Figure11_calibration.png"), width = 1600, height = 1600, res = 280)
plot(cal, xlab = "Predicted probability", ylab = "Observed probability",
     main = "Bootstrap-corrected calibration")
invisible(dev.off())
cat("Calibration plot saved.\n")

examples <- tibble::tribble(
  ~scenario,                   ~dbs, ~age_at_visit, ~duration_yrs, ~updrs3_score, ~LEDD, ~pre_mean, ~NP1DPRS, ~scopa, ~stai, ~pre_slope,
  "Low-risk DBS candidate",    1,    58,             2,             20,             300,   0,         0,        5,      55,    0.00,
  "Average DBS candidate",     1,    62,             5,             30,             800,   1,         1,        12,     65,    0.05,
  "High-risk DBS candidate",   1,    70,             8,             45,             1200,  2,         2,        20,     90,    0.15
)
examples$pred_prob <- predict(fit_lrm, newdata = examples, type = "fitted")
print(examples)
save_table(examples, "nomogram_example_predictions")
save_object(fit_lrm, "nomogram_model")


n=642   Mean absolute error=0.013   Mean squared error=0.00037
0.9 Quantile of absolute error=0.023



Calibration plot saved.


# A tibble: 3 × 12
  scenario     dbs age_at_visit duration_yrs updrs3_score  LEDD pre_mean NP1DPRS
  <chr>      <dbl>        <dbl>        <dbl>        <dbl> <dbl>    <dbl>   <dbl>
1 Low-risk …     1           58            2           20   300        0       0
2 Average D…     1           62            5           30   800        1       1
3 High-risk…     1           70            8           45  1200        2       2
# ℹ 4 more variables: scopa <dbl>, stai <dbl>, pre_slope <dbl>, pred_prob <dbl>
